# 02 — Dataset exploration and validation

**Estimated time:** 45 minutes<br>
**Prerequisites:** 01 — Dataset provenance and license<br>
**Learner-produced evidence:** a privacy-preserving quality report and dataset-card draft

## Learning objectives

- Measure missing, duplicate, label, length, and sensitive-pattern findings.
- Use the exact local tokenizer for token-length analysis.
- Interpret aggregate evidence without displaying unnecessary raw text.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## Why this matters

Training code cannot repair an unknown data contract. Missing fields, label imbalance, duplicates, long outliers, and sensitive strings all shape what a model can learn and what an evaluation can honestly claim. Exploration should therefore produce aggregate, repeatable evidence before it exposes raw examples or launches training.

## Key terms in plain language

- **schema:** the explicit contract for required fields, types, allowed values, and constraints.
- **label:** the expected answer or category attached to a supervised example.
- **label distribution:** the count or proportion of examples in each class; severe imbalance can make accuracy misleading.
- **duplicate:** a record whose normalized learning content is identical to another record.
- **near duplicate:** records that differ superficially but carry substantially the same learning signal.
- **token:** a unit produced by the model's tokenizer; it is not necessarily a word or character.
- **sensitive-data indicator:** a conservative pattern or classifier that flags possible sensitive content for review; it is not proof that content is or is not sensitive.
- **aggregate:** a count, distribution, or summary that reduces unnecessary exposure of individual rows.


## Mental model — how to think about this

Treat the dataset as the first model: it already encodes assumptions about what inputs exist, which answers count as correct, and which groups are common. Audit it as a system with inputs, invariants, and failure modes. Begin with schemas and aggregates, drill into a small masked sample only when a summary reveals something that needs explanation.

### Running example

Suppose one row says `"I forgot my password"` and another says `"  I FORGOT my password  "`. A versioned normalization rule may group them as duplicate learning content. If the two rows carry different intents, the conflict is evidence to quarantine—not permission to pick the convenient label.

### Questions to ask before continuing

- What properties are measured directly, and what properties are only heuristic signals?
- Which rare labels, lengths, languages, or sources could disappear in a single average?
- Could two records carry the same learning signal despite different punctuation or IDs?
- What is the minimum row-level content needed to investigate a problem safely?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **Validate the schema before statistics.** Invalid records should be counted and quarantined, not silently coerced into plausible values.
- **Inspect distributions and slices.** Report counts by label, length, language/source where verified, and policy-relevant subgroup rather than relying on a global row count.
- **Use the exact model tokenizer for final sizing.** Character or whitespace counts are useful proxies, but only the selected tokenizer determines context and training token lengths.
- **Detect duplicates before splitting.** Normalize with a versioned rule and group related records so they cannot leak across evidence boundaries.
- **Prefer aggregate-first privacy.** Mask bounded previews, never print whole datasets, and treat zero pattern matches as `none detected by this rule`, not a privacy guarantee.

## Common mistakes and why they fail

- **Printing random raw rows as exploration.** It increases exposure without first establishing what question the preview answers.
- **Treating a heuristic as ground truth.** Language, difficulty, toxicity, or sensitive-data rules have known blind spots and need explicit labels such as `detected` or `estimated`.
- **Deduplicating after the split.** A duplicate can already have crossed into validation or test.
- **Reporting only the average.** A healthy average can hide a missing class, an extreme tail, or a severe minority-slice failure.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Tool guidance:** [Hugging Face Dataset Cards documentation](https://huggingface.co/docs/hub/en/datasets-cards)
- **Risk guidance:** [NIST AI RMF Generative AI Profile](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence)


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Load the inspected local source

Reusable audit logic lives under `src/aai_local_finetuning/data`.
The notebook asks questions of the resulting evidence rather than
reimplementing the pipeline in cells.


In [ ]:
import json

from aai_local_finetuning.data import (
    PreparationConfig,
    audit_dataset,
    check_split_files,
    processing_source_sha256,
    sha256_file,
    summarize_instruction_tokens,
)
from aai_local_finetuning.settings import load_settings

settings = load_settings()
raw_csv = settings.csv_path
processed_dir = settings.processed_dir
if not raw_csv.is_file():
    raise FileNotFoundError(
        "The immutable Bitext CSV is missing. Prepare this machine online."
    )
manifest_path = processed_dir / "manifest.json"
quality_report_path = processed_dir / "quality_report.json"

## Quality audit

The record funnel explains why counts can shrink:

| Stage | Meaning |
|---|---|
| Source | Every parsed CSV row |
| Valid | Rows satisfying the required field and type contract |
| Unique | Repeated normalized learning content counted once |
| Non-conflicting | Duplicate groups whose labels do not disagree |
| Curated | Eligible records selected under the versioned balance policy |
| Split | Curated records assigned to train, validation, or frozen test |

The report contains counts and distributions, not raw samples. Exact
duplicates are measured after canonicalization. Near duplicates and
inferred templates are heuristic evidence and must be documented as such.

**Modern evidence practice:** preparation computes this expensive audit
once and puts its bytes under the dataset manifest. A lesson should verify
and read that evidence by default rather than silently repeat minutes of
work. Recompute when the source, policy, or pipeline changes, then compare
the new evidence before replacing the approved artifact. Set the switch
below to `True` when you deliberately want that full local recomputation.


In [ ]:
RECOMPUTE_FULL_AUDIT = False

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
processing_config = PreparationConfig.model_validate(manifest["processing"])
quality_evidence = manifest["artifacts"]["quality_report"]
source_evidence = next(
    item for item in manifest["raw_files"] if item["path"] == raw_csv.name
)
evidence_mismatches = []
if quality_report_path.stat().st_size != quality_evidence["size_bytes"]:
    evidence_mismatches.append("quality report size")
if sha256_file(quality_report_path) != quality_evidence["sha256"]:
    evidence_mismatches.append("quality report SHA-256")
if raw_csv.stat().st_size != source_evidence["size_bytes"]:
    evidence_mismatches.append("source size")
if sha256_file(raw_csv) != source_evidence["sha256"]:
    evidence_mismatches.append("source SHA-256")
processing_is_current = (
    processing_source_sha256() == manifest["processing_source_sha256"]
)
if not processing_is_current and not RECOMPUTE_FULL_AUDIT:
    evidence_mismatches.append("processing source SHA-256")
if evidence_mismatches:
    raise RuntimeError(
        "Prepared audit evidence is stale or damaged. Re-run flight "
        "preparation before studying. Failed checks: " + ", ".join(evidence_mismatches)
    )

prepared_audit_payload = json.loads(quality_report_path.read_text(encoding="utf-8"))
measured_fields = (
    "source_records",
    "valid_records",
    "unique_records",
    "curated_records",
    "invalid_record_count",
    "missing_by_field",
    "exact_duplicate_count",
    "exact_duplicate_rate",
    "near_duplicate_pairs",
    "near_duplicate_clusters",
    "conflicting_group_count",
    "excluded_conflicting_records",
)
if RECOMPUTE_FULL_AUDIT:
    audit_payload = audit_dataset(
        raw_csv,
        config=processing_config,
    ).model_dump(mode="json")
    audit_source = "recomputed now from immutable source bytes"
    changes_from_prepared = {
        key: {
            "prepared": prepared_audit_payload[key],
            "recomputed": audit_payload[key],
        }
        for key in measured_fields
        if audit_payload[key] != prepared_audit_payload[key]
    }
else:
    audit_payload = prepared_audit_payload
    audit_source = "prepared once and SHA-256 verified from the manifest"
    changes_from_prepared = {}
core_quality = {
    "evidence_source": audit_source,
    **{key: audit_payload[key] for key in measured_fields},
}
{
    "quality": core_quality,
    "changes_from_prepared": changes_from_prepared,
}

## Labels, lengths, language, and sensitive-looking patterns

Pattern matches are counts only. Email-, URL-, phone-like text and
placeholders are masked before portable training records are written.
Source flags remain explicit evaluation slices; difficulty is a separate
versioned heuristic, not a human quality label. Language below is the
source's declared coverage, not language detected independently in every row.


In [ ]:
token_lengths = summarize_instruction_tokens(raw_csv, settings.model_dir)
distribution_report = {
    "intents": audit_payload["intent_distribution"],
    "categories": audit_payload["category_distribution"],
    "declared_language_coverage": {
        settings.dataset.language: audit_payload["source_records"]
    },
    "instruction_characters": audit_payload["instruction_characters"],
    "instruction_word_proxy": audit_payload["instruction_words"],
    "pinned_tokenizer_tokens": token_lengths.model_dump(mode="json"),
    "source_flags": audit_payload["flag_distribution"],
    "difficulty": audit_payload["difficulty_distribution"],
    "sensitive_pattern_counts": audit_payload["sensitive_pattern_counts"],
}
distribution_report

## Preview the prepared split-integrity gate

Notebook 03 teaches how the split is constructed. For now, this is a
preview of already-prepared evidence: the gate examines boundaries
without displaying frozen test content. It checks exact, inferred-template,
and near-duplicate overlap plus target and demonstration leakage.


In [ ]:
integrity = check_split_files(processed_dir)
integrity.model_dump(mode="json")

## Dataset-card draft inputs

The tracked card remains reviewed prose. This draft makes measurements
easy to revisit whenever source bytes, processing, or split policy change.


In [ ]:
dataset_card_draft = {
    "source": manifest["dataset"],
    "fingerprint": manifest["dataset_fingerprint"],
    "quality": core_quality,
    "label_count": len(audit_payload["intent_distribution"]),
    "sensitive_review": audit_payload["sensitive_pattern_counts"],
    "split_strategy": manifest["split_strategy"],
    "review_required": [
        "near-duplicate threshold",
        "sensitive-looking content",
        "generated response policy",
        "redistribution obligations",
    ],
}
dataset_card_draft

## Exercise — identify the highest-risk assumption

Pick one measured finding and state what could go wrong if it were
ignored. Success means your answer connects a finding to either leakage,
fairness across labels, privacy, or misleading evaluation.


In [ ]:
finding = "near-duplicate clusters"
risk = (
    "Related templates crossing splits could make memorization look like "
    "generalization, so groups must stay inside one evidence boundary."
)
assert finding and len(risk.split()) >= 10
{"finding": finding, "risk": risk}

**Hint:** counts are not conclusions. Ask how each observation could bias
the final comparison or expose content unnecessarily.


## Checkpoint

The source has now been measured, not merely described. Keep the frozen
test content out of prompt design and model development.

**Next:** `03_leakage_safe_splits.ipynb` follows records from immutable
source to portable train, validation, and test boundaries.
